In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/DL/Transformer from Scratch

Mounted at /content/drive
/content/drive/MyDrive/DL/Transformer from Scratch


In [ ]:
# Thêm vào cell đầu tiên của notebook
%load_ext autoreload
%autoreload 2

In [ ]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 8.2 MB/s eta 0:00:00


In [ ]:
import os
print("Thư mục hiện tại:", os.getcwd())
print("Các file trong thư mục này:", os.listdir())

Thư mục hiện tại: /content/drive/MyDrive/DL/Transformer from Scratch
Các file trong thư mục này: ['Inference.ipynb', 'dataset.py', 'test.ipynb', 'preprocess.py', 'runs', '__pycache__', 'weights', '.ipynb_checkpoints', 'tokenizer_en.json', 'tokenizer_vi.json', 'dataset_cache_vi', 'config.py', 'training_chart.png', 'Finetune', 'RNNFinetune', 'medical_finetune', 'Data_Medi', 'train.py', 'model.py', 'main.ipynb', 'medcrab_1.5B.ipynb']


In [ ]:
!ls Finetune

config_finetune.py  med_dataset.py  train_finetune.py
DataMed.ipynb	    __pycache__     training_chart.png
FinetuneMed.ipynb   runs	    weights_finetune_300k


#MÔ HÌNH GỐC FINETUNE

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from tokenizers import Tokenizer
from datasets import load_from_disk
import sacrebleu
import math
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).parent))
from model import build_transformer
from dataset import causual_mask
from Finetune.med_dataset import MedTranslationDataset

# --- 2. CẤU HÌNH TỰ ĐỘNG (A100 vs T4) ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Đang chạy trên: {GPU_NAME}")

if "A100" in GPU_NAME:
    BATCH_SIZE = 256  # A100 chạy batch lớn
    DTYPE = torch.bfloat16
    print("Detected A100! Using Batch Size 256 & bfloat16")
else:
    BATCH_SIZE = 64  # T4 chạy batch nhỏ
    DTYPE = torch.float16
    print("Detected T4/Standard GPU. Using Batch Size 32 & float16")

CONFIG = {
    "seq_len": 350,
    "d_model": 512,
    "model_path": "./Finetune/weights_finetune_300k/finetune_best.pt",
    "data_path": "./Data_Medi/med_test",
    "tokenizer_en": "./tokenizer_en.json",
    "tokenizer_vi": "./tokenizer_vi.json",
}

# --- 3. LOAD DATA & MODEL ---
print("Loading Data & Model...")

# Load Tokenizers
tokenizer_src = Tokenizer.from_file(CONFIG['tokenizer_en'])
tokenizer_tgt = Tokenizer.from_file(CONFIG['tokenizer_vi'])
PAD_ID = tokenizer_tgt.token_to_id("[PAD]")

# Load Dataset
test_hf = load_from_disk(CONFIG['data_path'])
test_dataset = MedTranslationDataset(test_hf, tokenizer_src, tokenizer_tgt, CONFIG['seq_len'])

# DataLoader (Dùng default collate_fn là đủ vì dataset đã padding chuẩn)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# Load Model
model = build_transformer(
    tokenizer_src.get_vocab_size(),
    tokenizer_tgt.get_vocab_size(),
    CONFIG['seq_len'],
    CONFIG['d_model']
).to(DEVICE)

# Load Weights
ckpt = torch.load(CONFIG['model_path'], map_location=DEVICE)
state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
# Xử lý key _orig_mod nếu có
model.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in state_dict.items()})
model.eval()

# Compile model nếu là A100
if "A100" in GPU_NAME:
    print("Compiling model for A100...")
    model = torch.compile(model)

Đang chạy trên: Tesla T4
Detected T4/Standard GPU. Using Batch Size 32 & float16
Loading Data & Model...


In [ ]:
print(f"\nSTARTING LOSS EVALUATION ON {len(test_dataset)} SAMPLES...")

loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)
total_loss = 0
num_batches = 0

# Sử dụng Mixed Precision để tăng tốc tối đa
with torch.no_grad(), torch.autocast(device_type='cuda', dtype=DTYPE):
    for batch in tqdm(test_dataloader, desc="Calculating PPL"):
        # Move data
        encoder_input = batch['encoder_input'].to(DEVICE, non_blocking=True)
        decoder_input = batch['decoder_input'].to(DEVICE, non_blocking=True)
        encoder_mask = batch['encoder_mask'].to(DEVICE, non_blocking=True)
        decoder_mask = batch['decoder_mask'].to(DEVICE, non_blocking=True)
        label = batch['label'].to(DEVICE, non_blocking=True)

        # Forward
        encoder_output = model.encode(encoder_input, encoder_mask)
        decoder_output = model.decode(encoder_output, encoder_mask, decoder_input, decoder_mask)
        proj_output = model.project(decoder_output)

        # Loss
        loss = loss_fn(proj_output.view(-1, tokenizer_tgt.get_vocab_size()), label.view(-1))
        total_loss += loss.item()
        num_batches += 1

avg_loss = total_loss / num_batches
perplexity = math.exp(avg_loss)

print("\n" + "="*30)
print(f"REPORT (GPU: {GPU_NAME})")
print(f"Loss      : {avg_loss:.4f}")
print(f"Perplexity: {perplexity:.4f}")
print("="*30)

In [ ]:
# Hàm Greedy Decode
def greedy_decode(model, source, source_mask, tokenizer_tgt, max_len, device):
    sos_idx = tokenizer_tgt.token_to_id("[SOS]")
    eos_idx = tokenizer_tgt.token_to_id("[EOS]")
    encoder_output = model.encode(source, source_mask)
    decoder_input = torch.empty(1, 1).fill_(sos_idx).type_as(source).to(device)

    for _ in range(max_len):
        if decoder_input.size(1) >= max_len: break
        decoder_mask = causual_mask(decoder_input.size(1)).type_as(source_mask).to(device)
        out = model.decode(encoder_output, source_mask, decoder_input, decoder_mask)
        prob = model.project(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        decoder_input = torch.cat([decoder_input, torch.empty(1, 1).type_as(source).fill_(next_word.item()).to(device)], dim=1)
        if next_word.item() == eos_idx: break
    return decoder_input.squeeze(0)

# ---------------------------------------------------------
print(f"\nSTARTING BLEU EVALUATION (GPU: {GPU_NAME})...")

# None = test hết 16k câu
NUM_SAMPLES = None
# NUM_SAMPLES = 2000

indices = range(len(test_dataset)) if NUM_SAMPLES is None else range(NUM_SAMPLES)
print(f"Evaluating on {len(indices)} sentences...")

predictions = []
references = []

with torch.no_grad():
    for i in tqdm(indices, desc="Translating"):
        item = test_dataset[i]
        src = item['encoder_input'].unsqueeze(0).to(DEVICE)
        src_mask = item['encoder_mask'].unsqueeze(0).to(DEVICE)
        tgt_text = item['tgt_text']

        # Decode
        model_out = greedy_decode(model, src, src_mask, tokenizer_tgt, max_len=60, device=DEVICE)
        pred_text = tokenizer_tgt.decode(model_out.detach().cpu().numpy())

        predictions.append(pred_text)
        references.append(tgt_text)

# Tính BLEU
bleu = sacrebleu.corpus_bleu(predictions, [references])
print(f"\n>> BLEU Score ({len(indices)} samples): {bleu.score:.4f}")


STARTING BLEU EVALUATION (GPU: Tesla T4)...
Evaluating on 16581 sentences...


Translating: 100%|██████████| 16581/16581 [1:25:30<00:00,  3.23it/s]



>> BLEU Score (16581 samples): 38.0503


# MEDCRAB

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate datasets sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 57.6 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_from_disk
import sacrebleu
from tqdm import tqdm
import math

# -------------------------------------------------
# 1. CẤU HÌNH (CONFIG)
# -------------------------------------------------
MODEL_NAME = "pnnbao-ump/MedCrab-1.5b"
DATA_PATH = "./Data_Medi/med_test" # Folder chứa file arrow
MAX_SAMPLES = None  # Để None nếu muốn chạy hết. Để 100 nếu muốn test nhanh code.

# Tự động cấu hình GPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

if "A100" in GPU_NAME:
    BATCH_SIZE_PPL = 16  # Batch tính Loss (A100 mạnh nên để cao)
    BATCH_SIZE_GEN = 8   # Batch dịch
    DTYPE = torch.bfloat16
    print(f"Detected A100 ({GPU_NAME})! Using bfloat16 & High Batch Size")
else:
    BATCH_SIZE_PPL = 8   # T4 yếu hơn, để thấp tránh tràn VRAM
    BATCH_SIZE_GEN = 8
    DTYPE = torch.float16
    print(f"Detected Standard GPU ({GPU_NAME}). Using float16 & Low Batch Size")

# -------------------------------------------------
# 2. LOAD DATA & MODEL
# -------------------------------------------------
print("Loading Dataset...")
try:
    dataset = load_from_disk(DATA_PATH)
    if MAX_SAMPLES:
        dataset = dataset.select(range(MAX_SAMPLES))
    print(f"Loaded {len(dataset)} samples.")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit()

print("Loading Model (4-bit Quantization)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=DTYPE
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Mặc định right cho training/loss

# -------------------------------------------------
# 3. HÀM CHUẨN BỊ PROMPT (QUAN TRỌNG)
# -------------------------------------------------
def format_prompt(sample, for_generation=False):
    """
    Chuyển data Eng-Vie thành format hội thoại cho model.
    Format này phải khớp với cách model được train (thường là Alpaca style).
    """
    src = sample['Eng']
    tgt = sample['Vie']

    # Prompt Template
    prompt = f"### Instruction:\nDịch câu y khoa sau sang tiếng Việt.\n\n### Input:\n{src}\n\n### Response:\n"

    if for_generation:
        # Nếu đang dịch: Chỉ đưa câu hỏi
        return prompt
    else:
        # Nếu tính Loss: Đưa cả câu hỏi + câu trả lời + EOS
        return prompt + tgt + tokenizer.eos_token

print("PHASE 1: CALCULATING MASKED LOSS (ACCURATE)")

total_loss = 0
num_batches = 0

full_prompts = [format_prompt(x, for_generation=False) for x in dataset]

inputs_only = [format_prompt(x, for_generation=True) for x in dataset]

model.eval()
with torch.no_grad():
    for i in tqdm(range(0, len(full_prompts), BATCH_SIZE_PPL), desc="Computing Masked Loss"):
        # 1. Lấy batch
        batch_full = full_prompts[i : i + BATCH_SIZE_PPL]
        batch_input = inputs_only[i : i + BATCH_SIZE_PPL]

        # 2. Tokenize cả câu
        tokenized_full = tokenizer(
            batch_full,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(DEVICE)

        # 3. Tokenize riêng phần câu hỏi để biết nó dài bao nhiêu
        tokenized_input = tokenizer(
            batch_input,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
            add_special_tokens=False # Quan trọng: để tránh double BOS token
        )

        # 4. Tạo Labels
        labels = tokenized_full.input_ids.clone()

        for k in range(len(batch_full)):

            prompt_len = len(tokenizer.encode(batch_input[k], add_special_tokens=False))
            # Che phần prompt đi (gán -100)
            labels[k, :prompt_len] = -100
            # Che luôn cả phần padding
            labels[k][labels[k] == tokenizer.pad_token_id] = -100
        outputs = model(
            input_ids=tokenized_full.input_ids,
            attention_mask=tokenized_full.attention_mask,
            labels=labels
        )

        total_loss += outputs.loss.item()
        num_batches += 1

avg_loss = total_loss / num_batches
perplexity = math.exp(avg_loss)

print(f"\n>> Masked Loss: {avg_loss:.4f}")
print(f">> Perplexity : {perplexity:.4f}")

Detected A100 (NVIDIA A100-SXM4-40GB)! Using bfloat16 & High Batch Size
Loading Dataset...
Loaded 16581 samples.
Loading Model (4-bit Quantization)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

PHASE 1: CALCULATING MASKED LOSS (ACCURATE)


Computing Masked Loss: 100%|██████████| 1037/1037 [01:57<00:00,  8.86it/s]


>> Masked Loss: 1.4373
>> Perplexity : 4.2095


In [ ]:
MAX_SAMPLES = 2000
dataset = dataset.select(range(MAX_SAMPLES))

print("PHASE 2: GENERATING & CALCULATING BLEU")

predictions = []
references = []

# Đổi padding side sang trái để sinh từ (generation) chuẩn hơn
tokenizer.padding_side = "left"

prompts_gen = [format_prompt(x, for_generation=True) for x in dataset]
ground_truths = [x['Vie'] for x in dataset]

with torch.no_grad():
    for i in tqdm(range(0, len(prompts_gen), BATCH_SIZE_GEN), desc="Translating"):
        batch_prompts = prompts_gen[i : i + BATCH_SIZE_GEN]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(DEVICE)

        # Generate
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=128,   # Độ dài câu trả lời tối đa
            do_sample=False,      # Greedy decoding (ổn định nhất để chấm điểm)
            pad_token_id=tokenizer.eos_token_id
        )

        # Decode ra chữ
        decoded_batch = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        # Cắt bỏ phần prompt, chỉ lấy phần trả lời
        for text in decoded_batch:
            # Tìm đoạn sau "### Response:"
            if "### Response:" in text:
                answer = text.split("### Response:")[-1].strip()
            else:
                answer = text # Trường hợp model output lỗi
            predictions.append(answer)

references = [[r] for r in ground_truths] # Format cho sacrebleu

# In thử vài mẫu
print("\n--- Sample Results ---")
for k in range(min(3, len(predictions))):
    print(f"Input: {dataset[k]['Eng']}")
    print(f"Ref  : {ground_truths[k]}")
    print(f"Pred : {predictions[k]}")
    print("-" * 20)

# Tính BLEU
bleu = sacrebleu.corpus_bleu(predictions, references)
print(f"\n>> BLEU Score: {bleu.score:.2f}")

# Lưu kết quả
with open("medcrab_eval_results.txt", "w", encoding="utf-8") as f:
    f.write(f"Model: {MODEL_NAME}\n")
    f.write(f"Loss: {avg_loss:.4f}\n")
    f.write(f"PPL: {perplexity:.4f}\n")
    f.write(f"BLEU: {bleu.score:.2f}\n")

print("Done! Results saved to medcrab_eval_results.txt")

PHASE 2: GENERATING & CALCULATING BLEU


Translating: 100%|██████████| 250/250 [38:54<00:00,  9.34s/it]



--- Sample Results ---
Input: 43.1% of patients were treated with IVIG.
Ref  : 22 (43,1%) trẻ có điều trị IVIG.
Pred : 43.1% bệnh nhân được điều trị bằng IVIG.
--------------------
Input: The true question seems to be whether there is a unique type of diabetes related to direct viral toxicity.
Ref  : Câu hỏi thực sự là liệu có một loại đái tháo đường duy nhất liên quan đến độc tính trực tiếp của virus hay không.
Pred : Câu hỏi thực sự dường như là liệu có một loại đái tháo đường duy nhất liên quan đến độc tính trực tiếp của virus hay không.
--------------------
Input: Objectives: The study aimed to evaluate the criteria for quality of glomerular filtration of white blood cells with platelet nourishment solution prepared from top-bottom three (350ml) bag system at Hematology Blood Transfusion Hospital.
Ref  : Mục tiêu: Đánh giá các chỉ tiêu chất lượng khối tiểu cầu pool lọc bạch cầu có dung dịch nuôi dưỡng tiểu cầu được điều chế từ hệ thống túi ba (350 ml) đỉnh – đáy trên thiết bị tách

#MÔ HÌNH GỐC KHÔNG FINETUNE

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from tokenizers import Tokenizer
from datasets import load_from_disk
import sacrebleu
import math
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).parent))
from model import build_transformer
from dataset import causual_mask
from Finetune.med_dataset import MedTranslationDataset

# --- 2. CẤU HÌNH TỰ ĐỘNG (A100 vs T4) ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Đang chạy trên: {GPU_NAME}")

if "A100" in GPU_NAME:
    BATCH_SIZE = 256  # A100 chạy batch lớn
    DTYPE = torch.bfloat16
    print("Detected A100! Using Batch Size 256 & bfloat16")
else:
    BATCH_SIZE = 64  # T4 chạy batch nhỏ
    DTYPE = torch.float16
    print("Detected T4/Standard GPU. Using Batch Size 32 & float16")

CONFIG = {
    "seq_len": 350,
    "d_model": 512,
    "model_path": "./weights/tmodel_best.pt",
    "data_path": "./Data_Medi/med_test",
    "tokenizer_en": "./tokenizer_en.json",
    "tokenizer_vi": "./tokenizer_vi.json",
}

# --- 3. LOAD DATA & MODEL ---
print("Loading Data & Model...")

# Load Tokenizers
tokenizer_src = Tokenizer.from_file(CONFIG['tokenizer_en'])
tokenizer_tgt = Tokenizer.from_file(CONFIG['tokenizer_vi'])
PAD_ID = tokenizer_tgt.token_to_id("[PAD]")

# Load Dataset
test_hf = load_from_disk(CONFIG['data_path'])
test_dataset = MedTranslationDataset(test_hf, tokenizer_src, tokenizer_tgt, CONFIG['seq_len'])

# DataLoader (Dùng default collate_fn là đủ vì dataset đã padding chuẩn)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# Load Model
model = build_transformer(
    tokenizer_src.get_vocab_size(),
    tokenizer_tgt.get_vocab_size(),
    CONFIG['seq_len'],
    CONFIG['d_model']
).to(DEVICE)

# Load Weights
ckpt = torch.load(CONFIG['model_path'], map_location=DEVICE)
state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
# Xử lý key _orig_mod nếu có
model.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in state_dict.items()})
model.eval()

# Compile model nếu là A100
if "A100" in GPU_NAME:
    print("Compiling model for A100...")
    model = torch.compile(model)

Đang chạy trên: Tesla T4
Detected T4/Standard GPU. Using Batch Size 32 & float16
Loading Data & Model...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
print(f"\nSTARTING LOSS EVALUATION ON {len(test_dataset)} SAMPLES...")

loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)
total_loss = 0
num_batches = 0

# Sử dụng Mixed Precision để tăng tốc tối đa
with torch.no_grad(), torch.autocast(device_type='cuda', dtype=DTYPE):
    for batch in tqdm(test_dataloader, desc="Calculating PPL"):
        # Move data
        encoder_input = batch['encoder_input'].to(DEVICE, non_blocking=True)
        decoder_input = batch['decoder_input'].to(DEVICE, non_blocking=True)
        encoder_mask = batch['encoder_mask'].to(DEVICE, non_blocking=True)
        decoder_mask = batch['decoder_mask'].to(DEVICE, non_blocking=True)
        label = batch['label'].to(DEVICE, non_blocking=True)

        # Forward
        encoder_output = model.encode(encoder_input, encoder_mask)
        decoder_output = model.decode(encoder_output, encoder_mask, decoder_input, decoder_mask)
        proj_output = model.project(decoder_output)

        # Loss
        loss = loss_fn(proj_output.view(-1, tokenizer_tgt.get_vocab_size()), label.view(-1))
        total_loss += loss.item()
        num_batches += 1

avg_loss = total_loss / num_batches
perplexity = math.exp(avg_loss)

print("\n" + "="*30)
print(f"REPORT (GPU: {GPU_NAME})")
print(f"Loss      : {avg_loss:.4f}")
print(f"Perplexity: {perplexity:.4f}")
print("="*30)


STARTING LOSS EVALUATION ON 16581 SAMPLES...


Calculating PPL: 100%|██████████| 65/65 [00:22<00:00,  2.90it/s]


REPORT (GPU: NVIDIA A100-SXM4-80GB)
Loss      : 2.5997
Perplexity: 13.4593


In [ ]:
# Hàm Greedy Decode
def greedy_decode(model, source, source_mask, tokenizer_tgt, max_len, device):
    sos_idx = tokenizer_tgt.token_to_id("[SOS]")
    eos_idx = tokenizer_tgt.token_to_id("[EOS]")
    encoder_output = model.encode(source, source_mask)
    decoder_input = torch.empty(1, 1).fill_(sos_idx).type_as(source).to(device)

    for _ in range(max_len):
        if decoder_input.size(1) >= max_len: break
        decoder_mask = causual_mask(decoder_input.size(1)).type_as(source_mask).to(device)
        out = model.decode(encoder_output, source_mask, decoder_input, decoder_mask)
        prob = model.project(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        decoder_input = torch.cat([decoder_input, torch.empty(1, 1).type_as(source).fill_(next_word.item()).to(device)], dim=1)
        if next_word.item() == eos_idx: break
    return decoder_input.squeeze(0)

# ---------------------------------------------------------
print(f"\nSTARTING BLEU EVALUATION (GPU: {GPU_NAME})...")

# None = test hết 16k câu
NUM_SAMPLES = None
# NUM_SAMPLES = 2000

indices = range(len(test_dataset)) if NUM_SAMPLES is None else range(NUM_SAMPLES)
print(f"Evaluating on {len(indices)} sentences...")

predictions = []
references = []

with torch.no_grad():
    for i in tqdm(indices, desc="Translating"):
        item = test_dataset[i]
        src = item['encoder_input'].unsqueeze(0).to(DEVICE)
        src_mask = item['encoder_mask'].unsqueeze(0).to(DEVICE)
        tgt_text = item['tgt_text']

        # Decode
        model_out = greedy_decode(model, src, src_mask, tokenizer_tgt, max_len=60, device=DEVICE)
        pred_text = tokenizer_tgt.decode(model_out.detach().cpu().numpy())

        predictions.append(pred_text)
        references.append(tgt_text)

# Tính BLEU
bleu = sacrebleu.corpus_bleu(predictions, [references])
print(f"\n>> BLEU Score ({len(indices)} samples): {bleu.score:.4f}")


STARTING BLEU EVALUATION (GPU: Tesla T4)...
Evaluating on 16581 sentences...


Translating: 100%|██████████| 16581/16581 [1:31:19<00:00,  3.03it/s]



>> BLEU Score (16581 samples): 27.6632


#VINAI

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_from_disk
import sacrebleu
from tqdm import tqdm
import math

MODEL_NAME = "vinai/vinai-translate-en2vi"
DATA_PATH = "./Data_Medi/med_test"
MAX_SAMPLES = None # Để None để chạy hết, hoặc số nhỏ (vd 100) để test nhanh

# Tự động chọn thiết bị
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔹 Running on: {DEVICE}")

# VinAI Translate khá nhẹ (~4GB), T4 chạy vô tư với batch lớn hơn chút
BATCH_SIZE = 16

print("⏳ Loading Dataset...")
try:
    dataset = load_from_disk(DATA_PATH)
    if MAX_SAMPLES:
        dataset = dataset.select(range(MAX_SAMPLES))
    print(f"Loaded {len(dataset)} samples.")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit()

print(f"⏳ Loading Model {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="en_XX", tgt_lang="vi_VN")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

# Chuyển sang half-precision (float16) để chạy nhanh hơn và tiết kiệm VRAM
model.half()
model.eval()

🔹 Running on: cuda
⏳ Loading Dataset...
Loaded 16581 samples.
⏳ Loading Model vinai/vinai-translate-en2vi...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.88M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.79G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.79G [00:00<?, ?B/s]

MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(91408, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(91408, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)


In [ ]:
# -------------------------------------------------
# 3. TÍNH LOSS & PERPLEXITY (PPL)
# -------------------------------------------------
print("\n" + "="*40)
print("PHASE 1: CALCULATING LOSS & PPL")
print("="*40)

total_loss = 0
num_batches = 0
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=-100) # -100 để bỏ qua padding

src_texts = [x['Eng'] for x in dataset]
tgt_texts = [x['Vie'] for x in dataset]

# Batching thủ công
with torch.no_grad():
    for i in tqdm(range(0, len(src_texts), BATCH_SIZE), desc="Computing Loss"):
        batch_src = src_texts[i : i + BATCH_SIZE]
        batch_tgt = tgt_texts[i : i + BATCH_SIZE]

        # Tokenize Input (Tiếng Anh)
        inputs = tokenizer(
            batch_src,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(DEVICE)

        # Tokenize Output (Tiếng Việt) để làm Labels
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                batch_tgt,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).input_ids.to(DEVICE)

        # Thay thế pad_token_id bằng -100 để không tính loss vào padding
        labels[labels == tokenizer.pad_token_id] = -100

        # Forward pass
        # Model Seq2Seq tự động shift labels để tạo decoder_input_ids bên trong
        outputs = model(**inputs, labels=labels)

        loss = outputs.loss
        total_loss += loss.item()
        num_batches += 1

avg_loss = total_loss / num_batches
perplexity = math.exp(avg_loss)

print(f"\n>> Average Loss: {avg_loss:.4f}")
print(f">> Perplexity  : {perplexity:.4f}")


PHASE 1: CALCULATING LOSS & PPL


Computing Loss:   0%|          | 0/1037 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Computing Loss: 100%|██████████| 1037/1037 [00:30<00:00, 33.78it/s]


>> Average Loss: 1.2808
>> Perplexity  : 3.5994


In [ ]:
print("PHASE 2: GENERATING & CALCULATING BLEU")

src_texts = [x['Eng'] for x in dataset]
tgt_texts = [x['Vie'] for x in dataset]

predictions = []
references = [[t] for t in tgt_texts]

with torch.no_grad():
    for i in tqdm(range(0, len(src_texts), BATCH_SIZE), desc="Translating"):
        batch_src = src_texts[i : i + BATCH_SIZE]

        inputs = tokenizer(
            batch_src,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(DEVICE)

        # Generate
        # forced_bos_token_id: Bắt buộc output bắt đầu bằng token tiếng Việt
        generated_ids = model.generate(
            **inputs,
            max_length=128,
            forced_bos_token_id=tokenizer.lang_code_to_id["vi_VN"]
        )

        decoded_batch = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        predictions.extend(decoded_batch)

# In thử mẫu
print("\n--- Sample Results ---")
for k in range(min(3, len(predictions))):
    print(f"Src : {src_texts[k]}")
    print(f"Ref : {tgt_texts[k]}")
    print(f"Pred: {predictions[k]}")
    print("-" * 20)

# Tính BLEU
bleu = sacrebleu.corpus_bleu(predictions, references)


In [ ]:
print(f"\n>> BLEU Score: {bleu.score:.2f}")


>> BLEU Score: 38.99
